####  PCA (Principal Component Analysis) is used to reduce the number of features and then compares seven Machine Learning algorithms to find which one predicts the wine categories most accurately.

In [1]:
# Importing the libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (accuracy_score,classification_report,confusion_matrix)

# Importing the dataset
dataset=pd.read_csv('Wine.csv')
#This extracts the first 13 columns(0to12) as input features.: means select all rows.
#values converts the selected data into a NumPy array.
X = dataset.iloc[:, 0:13].values
#This extracts the column at position 13 as the target/Output.
y = dataset.iloc[:, 13].values


In [2]:
# Splitting the dataset into the Training set and Test set
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 0)

# Feature Scaling
from sklearn.preprocessing import StandardScaler
sc = StandardScaler()
X_train = sc.fit_transform(X_train)
X_test = sc.transform(X_test)

#PCA is a dimensionality-reduction technique. It converts a large set of original features into a smaller set of 
#new features called principal components.
def apply_pca(X_train, X_test, n):
    pca = PCA(n_components=n)
    #This performs two operations:
    #1. fit: Learns the principal directions from the training data.
    #2. transform: Converts the original training features into principal components.
    #The original 13 features are transformed into 2 components.
    #Important: PC1 and PC2 are combinations of the original features. They are not simply the first two original columns.
    #This tells you how much of the data's variation each component explains.
    explained_variance = pca.explained_variance_ratio_
    X_train_pca = pca.fit_transform(X_train)
    X_test_pca = pca.transform(X_test)  
    print("Number of original features:", X_train.shape[1])
    print("Number of PCA components:", X_train_pca.shape[1])
    print("Explained variance ratio:", explained_variance)
    print("Total explained variance:", sum(explained_variance))
    return X_train_pca, X_test_pca, pca

#Apply PCA with 2 componants
X_train_pca, X_test_pca, pca = apply_pca(X_train, X_test, 2)


Number of original features: 13
Number of PCA components: 2
Explained variance ratio: [0.36884109 0.19318394]
Total explained variance: 0.562025035922731


In [3]:
#Check how well the trained model predicts the test data.   
def cm_prediction(classifier, X_test, y_test):
    y_pred = classifier.predict(X_test)

    from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
    #A confusion matrix compares actual categories with predicted categories. It helps you see which classes are predicted 
    #correctly or confused with others.
    cm = confusion_matrix(y_test, y_pred)
    #Accuracy is the proportion of test samples predicted correctly.
    Accuracy = accuracy_score(y_test, y_pred)
    #This report provides: Precision: How many predictions for a class were correct.
    #Recall: How many actual samples of a class were correctly identified.
    #F1-score: A combined measure of precision and recall.
    #Support: The number of actual samples in each class.
    report = classification_report(y_test, y_pred)
    return classifier, Accuracy, report, X_test, y_test, cm



In [4]:
#Logistic Regression estimates the probabilities of different classes and predicts the most suitable class.
def logistic(X_train, y_train, X_test, y_test):

    from sklearn.linear_model import LogisticRegression
    classifier = LogisticRegression(random_state=0)
    classifier.fit(X_train, y_train)
    classifier, Accuracy, report, X_test, y_test, cm = cm_prediction(classifier, X_test, y_test)
    return classifier, Accuracy, report, X_test, y_test, cm

#A linear Support Vector Machine attempts to separate classes using linear decision boundaries.
def svm_linear(X_train, y_train, X_test, y_test):

    from sklearn.svm import SVC
    classifier = SVC(kernel='linear', random_state=0)
    classifier.fit(X_train, y_train)
    return cm_prediction(classifier, X_test, y_test)

#This uses the RBF (Radial Basis Function) kernel.
#It can learn more complex, curved decision boundaries when a straight boundary is insufficient.
def svm_NL(X_train, y_train, X_test, y_test):

    classifier = SVC(kernel="rbf", random_state=0)
    classifier.fit(X_train, y_train)
    return cm_prediction(classifier, X_test, y_test)

#Gaussian Naive Bayes uses probability to predict the class.
#It assumes that each feature follows a Gaussian distribution within each class and that features are conditionally 
#independent given the class.
def Navie(X_train, y_train, X_test, y_test):

    classifier = GaussianNB()
    classifier.fit(X_train, y_train)
    return cm_prediction(classifier, X_test, y_test)

#KNN predicts a sample's class based on nearby training samples.
#n_neighbors=5: Uses five neighbors.
#metric="minkowski": Specifies the distance metric.
#p=2: Uses Euclidean distance for this metric.
def knn(X_train, y_train, X_test, y_test):

    classifier = KNeighborsClassifier(n_neighbors=5,metric="minkowski",p=2)
    classifier.fit(X_train, y_train)
    return cm_prediction(classifier, X_test, y_test)

#A Decision Tree makes a sequence of decisions to classify each sample.
#criterion="entropy": Uses entropy to evaluate splits.
#random_state=0: Helps make the results reproducible when the algorithm uses randomness.
def Decision(X_train, y_train, X_test, y_test):

    classifier = DecisionTreeClassifier(criterion="entropy",random_state=0)
    classifier.fit(X_train, y_train)
    return cm_prediction(classifier, X_test, y_test)

#Random Forest builds multiple decision trees and combines their predictions.
#n_estimators=10: Builds 10 trees.
#criterion="entropy": Uses entropy to evaluate tree splits.
#random_state=0: Helps make the results reproducible.
def random(X_train, y_train, X_test, y_test):

    classifier = RandomForestClassifier(n_estimators=10,criterion="entropy",random_state=0)
    classifier.fit(X_train, y_train)
    return cm_prediction(classifier, X_test, y_test)



In [5]:
# Store Accuracy Results
#You want to store the accuracy of each algorithm so you can compare all seven models in one table.
acclog = []
accsvml = []
accsvmnl = []
accknn = []
accnav = []
accdes = []
accrf = []

#Here, append() adds the accuracy value to the list.
# Logistic Regression
classifier, Accuracy, report, _,_, cm = logistic(X_train_pca, y_train, X_test_pca, y_test)
acclog.append(Accuracy)


# Linear SVM
classifier, Accuracy, report, _, _, cm = svm_linear(X_train_pca, y_train, X_test_pca, y_test)
accsvml.append(Accuracy)


# Non-linear SVM
classifier, Accuracy, report, _, _, cm = svm_NL(X_train_pca, y_train, X_test_pca, y_test)
accsvmnl.append(Accuracy)


# KNN
classifier, Accuracy, report, _, _, cm = knn(X_train_pca, y_train, X_test_pca, y_test)
accknn.append(Accuracy)


# Naive Bayes
classifier, Accuracy, report, _, _, cm = Navie(X_train_pca, y_train, X_test_pca, y_test)
accnav.append(Accuracy)


# Decision Tree
classifier, Accuracy, report, _, _, cm = Decision(X_train_pca, y_train, X_test_pca, y_test)
accdes.append(Accuracy)


# Random Forest
classifier, Accuracy, report, _, _, cm = random(X_train_pca, y_train, X_test_pca, y_test)
accrf.append(Accuracy)



In [7]:
#This creates a function that receives seven lists containing accuracy values.
def selectk_Classification(acclog, accsvml, accsvmnl,accknn, accnav, accdes, accrf):
    dataframe = pd.DataFrame(index=["PCA"],columns=["Logistic","SVMl","SVMnl","KNN","Navie","Decision","Random"],
                             dtype=float)

    #The [0] means retrieve the first element from each list.
    #One limitation: Your function assumes every accuracy list contains at least one value. 
    #If a list is empty, accessing [0] will produce an IndexError.
    dataframe.loc["PCA"] = [acclog[0],accsvml[0],accsvmnl[0],accknn[0],accnav[0],accdes[0],accrf[0]]
    return dataframe

# Create the accuracy comparison table
result = selectk_Classification(acclog,accsvml,accsvmnl,accknn,accnav,accdes,accrf)

# Display the accuracy table
print("\nAccuracy Comparison Table:")
print(result)

# Find the best algorithm.
#result.iloc[0] selects the first row of the table.
#.idxmax() finds the column name containing the highest value.
best_algorithm = result.iloc[0].idxmax()
best_accuracy = result.iloc[0].max()

print("\nBest Algorithm:", best_algorithm)
print("Best Accuracy:", round(best_accuracy, 4))

# Display explained variance
print("\nTotal variance explained by 2 PCA components:",round(sum(pca.explained_variance_ratio_), 4))


Accuracy Comparison Table:
     Logistic      SVMl     SVMnl       KNN     Navie  Decision    Random
PCA  0.972222  0.972222  0.972222  0.972222  0.972222  0.972222  0.972222

Best Algorithm: Logistic
Best Accuracy: 0.9722

Total variance explained by 2 PCA components: 0.562
